## Étape 6 : données simulées

Ce notebook génère des devis, réservations et tickets fictifs à partir
de **vraies lignes de prix** de `PRICELINES_BAF09`.

Trois types de cas fabriqués :
- **A) Explicable** : ligne avec version système + Manual → `evidence=True`
- **B) À escalader** : écart de prix sans preuve dans les données → `evidence=False`
- **C) Ambigu** : route avec dry ET reefer, sans type dans le ticket

Le fret de base est inventé (il n'existe dans aucun fichier fourni).

**Ne relancer ce notebook qu'une fois.** Les IDs sont fixes.

In [ ]:
import pandas as pd
from snowflake.snowpark.context import get_active_session
from datetime import date

session = get_active_session()

### 1. Devis (QUOTES)

Chaque devis référence un `detail_uid` existant dans `PRICELINES_BAF09`.
Le fret de base est inventé. Le BAF09 vient des vraies données.

In [ ]:
quotes = pd.DataFrame([
    # === CAS A : explicable (evidence=True) ===
    # MXVER→ESSCT : detail_uid 360146852 a CTB INCLUDED + Manual ADDITIONAL 75/150/150
    {
        'QUOTE_ID': 'QSPOT-A1',
        'DETAIL_UID': 360146852,
        'POL_CODE': 'MXVER', 'POD_CODE': 'ESSCT',
        'CONTAINER_TYPE': 'ST', 'CONTAINER_SIZE': '40ST',
        'QUOTE_DATE': date(2026, 4, 10),
        'BASE_FREIGHT': 1200, 'BAF09_AMOUNT': 150, 'TOTAL': 1350,
        'CASE_TYPE': 'A_EXPLICABLE',
        'EXPECTED_DECISION': 'repondre',
        'EXPECTED_EVIDENCE': True,
        'NOTE': 'BAF09 modifié manuellement de INCLUDED à ADDITIONAL 150. Le client voit 1350 au lieu de 1200.'
    },
    # ALDRZ→AEJEA : detail_uid 362639043 a CTM INCLUDED + Manual ADDITIONAL 75/150
    {
        'QUOTE_ID': 'QSPOT-A2',
        'DETAIL_UID': 362639043,
        'POL_CODE': 'ALDRZ', 'POD_CODE': 'AEJEA',
        'CONTAINER_TYPE': 'ST', 'CONTAINER_SIZE': '20ST',
        'QUOTE_DATE': date(2026, 3, 20),
        'BASE_FREIGHT': 800, 'BAF09_AMOUNT': 75, 'TOTAL': 875,
        'CASE_TYPE': 'A_EXPLICABLE',
        'EXPECTED_DECISION': 'repondre',
        'EXPECTED_EVIDENCE': True,
        'NOTE': 'BAF09 modifié manuellement de INCLUDED à ADDITIONAL 75.'
    },
    # ALDRZ→AEAJM : detail_uid 362639033
    {
        'QUOTE_ID': 'QSPOT-A3',
        'DETAIL_UID': 362639033,
        'POL_CODE': 'ALDRZ', 'POD_CODE': 'AEAJM',
        'CONTAINER_TYPE': 'ST', 'CONTAINER_SIZE': '40HC',
        'QUOTE_DATE': date(2026, 4, 1),
        'BASE_FREIGHT': 950, 'BAF09_AMOUNT': 150, 'TOTAL': 1100,
        'CASE_TYPE': 'A_EXPLICABLE',
        'EXPECTED_DECISION': 'repondre',
        'EXPECTED_EVIDENCE': True,
        'NOTE': 'BAF09 modifié manuellement. 40HC à 150 USD.'
    },

    # === CAS B : à escalader (evidence=False) ===
    # PKKHI→LVRIX : detail_uid 359836078, une seule version CTB, pas de Manual
    {
        'QUOTE_ID': 'QSPOT-B1',
        'DETAIL_UID': 359836078,
        'POL_CODE': 'PKKHI', 'POD_CODE': 'LVRIX',
        'CONTAINER_TYPE': 'ST', 'CONTAINER_SIZE': '40ST',
        'QUOTE_DATE': date(2026, 3, 25),
        'BASE_FREIGHT': 2200, 'BAF09_AMOUNT': 300, 'TOTAL': 2500,
        'CASE_TYPE': 'B_ESCALADE',
        'EXPECTED_DECISION': 'escalader',
        'EXPECTED_EVIDENCE': False,
        'NOTE': 'Le client conteste le total mais la BAF09 n\'a qu\'une version système. Pas de preuve d\'écart.'
    },
    # BDCGP→MAAGA : detail_uid 359468845, seul CTB
    {
        'QUOTE_ID': 'QSPOT-B2',
        'DETAIL_UID': 359468845,
        'POL_CODE': 'BDCGP', 'POD_CODE': 'MAAGA',
        'CONTAINER_TYPE': 'ST', 'CONTAINER_SIZE': '20ST',
        'QUOTE_DATE': date(2026, 3, 18),
        'BASE_FREIGHT': 1500, 'BAF09_AMOUNT': 150, 'TOTAL': 1650,
        'CASE_TYPE': 'B_ESCALADE',
        'EXPECTED_DECISION': 'escalader',
        'EXPECTED_EVIDENCE': False,
        'NOTE': 'BAF09 ADDITIONAL 150 en version système uniquement. Pas de preuve d\'un changement manuel.'
    },
    # IDKUM→GBTIL : detail_uid 362353646, seul CTB
    {
        'QUOTE_ID': 'QSPOT-B3',
        'DETAIL_UID': 362353646,
        'POL_CODE': 'IDKUM', 'POD_CODE': 'GBTIL',
        'CONTAINER_TYPE': 'ST', 'CONTAINER_SIZE': '40HC',
        'QUOTE_DATE': date(2026, 3, 25),
        'BASE_FREIGHT': 1800, 'BAF09_AMOUNT': 300, 'TOTAL': 2100,
        'CASE_TYPE': 'B_ESCALADE',
        'EXPECTED_DECISION': 'escalader',
        'EXPECTED_EVIDENCE': False,
        'NOTE': 'Le client attendait un BAF inclus mais le système a ADDITIONAL 300. Pas de version Manual.'
    },

    # === CAS C : ambigu (dry + reefer sur la même route) ===
    # AEJEA→AUADL : 2 detail_uids (24804-2 en ST, 24333-4 en RF)
    {
        'QUOTE_ID': 'QSPOT-C1',
        'DETAIL_UID': 363279621,  # la ligne ST
        'POL_CODE': 'AEJEA', 'POD_CODE': 'AUADL',
        'CONTAINER_TYPE': None, 'CONTAINER_SIZE': '40ST',
        'QUOTE_DATE': date(2026, 3, 20),
        'BASE_FREIGHT': 1600, 'BAF09_AMOUNT': 0, 'TOTAL': 1600,
        'CASE_TYPE': 'C_AMBIGU',
        'EXPECTED_DECISION': 'preciser',
        'EXPECTED_EVIDENCE': False,
        'NOTE': 'Route avec ligne ST et RF. Sans le type de conteneur, impossible de trancher.'
    },
    # ALDRZ→ASPPG : ST (348206737) + RF (348207884)
    {
        'QUOTE_ID': 'QSPOT-C2',
        'DETAIL_UID': 348206737,  # la ligne ST
        'POL_CODE': 'ALDRZ', 'POD_CODE': 'ASPPG',
        'CONTAINER_TYPE': None, 'CONTAINER_SIZE': '20ST',
        'QUOTE_DATE': date(2026, 3, 15),
        'BASE_FREIGHT': 3200, 'BAF09_AMOUNT': 0, 'TOTAL': 3200,
        'CASE_TYPE': 'C_AMBIGU',
        'EXPECTED_DECISION': 'preciser',
        'EXPECTED_EVIDENCE': False,
        'NOTE': 'Route avec ligne dry et reefer. Le ticket ne précise pas le type.'
    },

    # === CAS MIXTES pour variété ===
    # Cas A avec un montant différent
    {
        'QUOTE_ID': 'QSPOT-A4',
        'DETAIL_UID': 360147399,
        'POL_CODE': 'MXVER', 'POD_CODE': 'ESSCT',
        'CONTAINER_TYPE': 'ST', 'CONTAINER_SIZE': '20ST',
        'QUOTE_DATE': date(2026, 4, 12),
        'BASE_FREIGHT': 900, 'BAF09_AMOUNT': 75, 'TOTAL': 975,
        'CASE_TYPE': 'A_EXPLICABLE',
        'EXPECTED_DECISION': 'repondre',
        'EXPECTED_EVIDENCE': True,
        'NOTE': 'Même route MXVER-ESSCT, autre detail_uid, même écart CTB/Manual.'
    },
    # Cas B sans devis lié (ticket directement)
    {
        'QUOTE_ID': 'QSPOT-B4',
        'DETAIL_UID': 362351308,
        'POL_CODE': 'JPYOK', 'POD_CODE': 'PLSZZ',
        'CONTAINER_TYPE': 'ST', 'CONTAINER_SIZE': '40ST',
        'QUOTE_DATE': date(2026, 3, 28),
        'BASE_FREIGHT': 2000, 'BAF09_AMOUNT': 300, 'TOTAL': 2300,
        'CASE_TYPE': 'B_ESCALADE',
        'EXPECTED_DECISION': 'escalader',
        'EXPECTED_EVIDENCE': False,
        'NOTE': 'Yokohama→Szczecin. BAF système à 300, pas de version Manual.'
    },
])

print(f"QUOTES : {len(quotes)} devis")
print(f"  Cas A (explicable) : {(quotes.CASE_TYPE == 'A_EXPLICABLE').sum()}")
print(f"  Cas B (escalade)   : {(quotes.CASE_TYPE == 'B_ESCALADE').sum()}")
print(f"  Cas C (ambigu)     : {(quotes.CASE_TYPE == 'C_AMBIGU').sum()}")
quotes[['QUOTE_ID','POL_CODE','POD_CODE','CONTAINER_TYPE','TOTAL','CASE_TYPE','EXPECTED_DECISION']]

### 2. Réservations (BOOKINGS)

Certains devis donnent lieu à une réservation. Chaque booking référence un devis existant.

In [ ]:
bookings = pd.DataFrame([
    {
        'BOOKING_ID': 'BKG-001',
        'QUOTE_ID': 'QSPOT-A1',
        'BOOKING_DATE': date(2026, 4, 11),
        'CONTAINER_COUNT': 2,
        'STATUS': 'CONFIRMED',
        'CUSTOMER_REF': 'CLI-4521',
        'NOTE': 'Booking confirmé sur le devis A1 (MXVER→ESSCT). Le client a accepté le prix.'
    },
    {
        'BOOKING_ID': 'BKG-002',
        'QUOTE_ID': 'QSPOT-A2',
        'BOOKING_DATE': date(2026, 3, 22),
        'CONTAINER_COUNT': 1,
        'STATUS': 'CONFIRMED',
        'CUSTOMER_REF': 'CLI-7803',
        'NOTE': 'Booking sur devis A2 (ALDRZ→AEJEA). Le client n\'a pas encore vu l\'écart de prix.'
    },
    {
        'BOOKING_ID': 'BKG-003',
        'QUOTE_ID': 'QSPOT-B1',
        'BOOKING_DATE': date(2026, 3, 26),
        'CONTAINER_COUNT': 3,
        'STATUS': 'CONFIRMED',
        'CUSTOMER_REF': 'CLI-1190',
        'NOTE': 'Booking sur B1 (PKKHI→LVRIX). Le client conteste la facture après livraison.'
    },
    {
        'BOOKING_ID': 'BKG-004',
        'QUOTE_ID': 'QSPOT-B2',
        'BOOKING_DATE': date(2026, 3, 19),
        'CONTAINER_COUNT': 1,
        'STATUS': 'CANCELLED',
        'CUSTOMER_REF': 'CLI-3345',
        'NOTE': 'Booking annulé suite à un désaccord sur le prix.'
    },
    {
        'BOOKING_ID': 'BKG-005',
        'QUOTE_ID': 'QSPOT-C1',
        'BOOKING_DATE': date(2026, 3, 21),
        'CONTAINER_COUNT': 5,
        'STATUS': 'PENDING',
        'CUSTOMER_REF': 'CLI-8877',
        'NOTE': 'Booking en attente, le type de conteneur n\'est pas confirmé.'
    },
    {
        'BOOKING_ID': 'BKG-006',
        'QUOTE_ID': 'QSPOT-A4',
        'BOOKING_DATE': date(2026, 4, 14),
        'CONTAINER_COUNT': 1,
        'STATUS': 'CONFIRMED',
        'CUSTOMER_REF': 'CLI-2210',
        'NOTE': 'Booking sur A4. Le client ouvre un ticket après la facture.'
    },
])

print(f"BOOKINGS : {len(bookings)} réservations")
print(f"  CONFIRMED : {(bookings.STATUS == 'CONFIRMED').sum()}")
print(f"  CANCELLED : {(bookings.STATUS == 'CANCELLED').sum()}")
print(f"  PENDING   : {(bookings.STATUS == 'PENDING').sum()}")
bookings[['BOOKING_ID','QUOTE_ID','STATUS','CUSTOMER_REF']]

### 3. Tickets de support (TICKETS)

Chaque ticket a une description réaliste et une réponse attendue notée.

In [ ]:
tickets = pd.DataFrame([
    # === CAS A : explicable ===
    {
        'TICKET_ID': 'TK-SIM-001',
        'QUOTE_ID': 'QSPOT-A1',
        'BOOKING_ID': 'BKG-001',
        'CREATED_DATE': date(2026, 4, 15),
        'DESCRIPTION': 'The BAF09 surcharge on my booking BKG-001 for MXVER to ESSCT shows 150 USD per 40ST. '
                       'I was told the BAF would be included in the ocean freight. Why am I being charged extra?',
        'CASE_TYPE': 'A_EXPLICABLE',
        'EXPECTED_DECISION': 'repondre',
        'EXPECTED_FINDING': 'La ligne 2378-134 (detail_uid 360146852) a été modifiée manuellement : '
                           'BAF09 passé de INCLUDED à ADDITIONAL 75/150/150.',
        'EXPECTED_ANSWER': 'Le BAF09 a été modifié manuellement de INCLUDED à ADDITIONAL sur cette route. '
                          'Le montant de 150 USD/40ST est la surcharge appliquée après modification.'
    },
    {
        'TICKET_ID': 'TK-SIM-002',
        'QUOTE_ID': 'QSPOT-A2',
        'BOOKING_ID': 'BKG-002',
        'CREATED_DATE': date(2026, 3, 24),
        'DESCRIPTION': 'Hello, my quote QSPOT-A2 for ALDRZ to AEJEA has a BAF09 of 75 USD on a 20ST container. '
                       'The system should show INCLUDED. Can you check?',
        'CASE_TYPE': 'A_EXPLICABLE',
        'EXPECTED_DECISION': 'repondre',
        'EXPECTED_FINDING': 'La ligne 222131-3 (detail_uid 362639043) existe en version CTM (INCLUDED) '
                           'et en version Manual (ADDITIONAL 75/150).',
        'EXPECTED_ANSWER': 'Le BAF09 a été modifié manuellement sur cette route de INCLUDED à ADDITIONAL 75 USD/20ST.'
    },
    {
        'TICKET_ID': 'TK-SIM-003',
        'QUOTE_ID': 'QSPOT-A4',
        'BOOKING_ID': 'BKG-006',
        'CREATED_DATE': date(2026, 4, 16),
        'DESCRIPTION': 'I received my invoice for booking BKG-006 on MXVER-ESSCT and there is a BAF09 charge '
                       'of 75 USD that was not in the original rate. Please explain.',
        'CASE_TYPE': 'A_EXPLICABLE',
        'EXPECTED_DECISION': 'repondre',
        'EXPECTED_FINDING': 'Ligne avec version CTB INCLUDED et Manual ADDITIONAL.',
        'EXPECTED_ANSWER': 'Le BAF09 a été ajouté manuellement comme surcharge additionnelle de 75 USD/20ST.'
    },

    # === CAS B : à escalader ===
    {
        'TICKET_ID': 'TK-SIM-004',
        'QUOTE_ID': 'QSPOT-B1',
        'BOOKING_ID': 'BKG-003',
        'CREATED_DATE': date(2026, 4, 2),
        'DESCRIPTION': 'We shipped 3 containers PKKHI to LVRIX and the BAF09 was charged at 300 USD/40ST. '
                       'Our contract says BAF should be 200 USD maximum. Please investigate.',
        'CASE_TYPE': 'B_ESCALADE',
        'EXPECTED_DECISION': 'escalader',
        'EXPECTED_FINDING': '',
        'EXPECTED_ANSWER': 'Pas de preuve dans les données BAF09 d\'un écart entre version système et manuelle. '
                          'Le ticket est transféré à l\'équipe pricing pour investigation.'
    },
    {
        'TICKET_ID': 'TK-SIM-005',
        'QUOTE_ID': 'QSPOT-B2',
        'BOOKING_ID': 'BKG-004',
        'CREATED_DATE': date(2026, 3, 20),
        'DESCRIPTION': 'Hello Team, the total for BDCGP to MAAGA is much higher than expected. '
                       'I cancelled my booking because the price doesn\'t match what I was quoted verbally.',
        'CASE_TYPE': 'B_ESCALADE',
        'EXPECTED_DECISION': 'escalader',
        'EXPECTED_FINDING': '',
        'EXPECTED_ANSWER': 'La ligne BAF09 n\'a qu\'une version système. L\'écart avec le prix verbal '
                          'ne peut pas être vérifié dans les données. Escalade nécessaire.'
    },
    {
        'TICKET_ID': 'TK-SIM-006',
        'QUOTE_ID': 'QSPOT-B3',
        'BOOKING_ID': None,
        'CREATED_DATE': date(2026, 3, 27),
        'DESCRIPTION': 'Hi, for the quote IDKUM to GBTIL I see a BAF of 300 USD on 40HC. '
                       'Last month it was included. Why the change?',
        'CASE_TYPE': 'B_ESCALADE',
        'EXPECTED_DECISION': 'escalader',
        'EXPECTED_FINDING': '',
        'EXPECTED_ANSWER': 'La BAF09 est en version système ADDITIONAL 300. Il n\'y a pas de preuve '
                          'd\'une modification manuelle. Transfert à l\'équipe.'
    },

    # === CAS C : ambigu ===
    {
        'TICKET_ID': 'TK-SIM-007',
        'QUOTE_ID': 'QSPOT-C1',
        'BOOKING_ID': 'BKG-005',
        'CREATED_DATE': date(2026, 3, 22),
        'DESCRIPTION': 'I need to check the BAF09 for my shipment AEJEA to AUADL. '
                       'The quote shows no surcharge but I want to confirm before booking.',
        'CASE_TYPE': 'C_AMBIGU',
        'EXPECTED_DECISION': 'preciser',
        'EXPECTED_FINDING': '',
        'EXPECTED_ANSWER': 'Deux lignes existent pour cette route : une dry (ST) et une reefer (RF). '
                          'Merci de préciser le type de conteneur pour identifier la bonne ligne.'
    },
    {
        'TICKET_ID': 'TK-SIM-008',
        'QUOTE_ID': 'QSPOT-C2',
        'BOOKING_ID': None,
        'CREATED_DATE': date(2026, 3, 16),
        'DESCRIPTION': 'Hello, I\'m shipping from ALDRZ to ASPPG and I don\'t see any BAF09 on my offer. '
                       'Is the BAF really included or is it a system error?',
        'CASE_TYPE': 'C_AMBIGU',
        'EXPECTED_DECISION': 'preciser',
        'EXPECTED_FINDING': '',
        'EXPECTED_ANSWER': 'La route ALDRZ→ASPPG a une ligne dry et une ligne reefer, toutes deux avec BAF INCLUDED. '
                          'Merci de confirmer le type de conteneur pour valider.'
    },

    # === CAS SUPPLEMENTAIRES ===
    {
        'TICKET_ID': 'TK-SIM-009',
        'QUOTE_ID': 'QSPOT-B4',
        'BOOKING_ID': None,
        'CREATED_DATE': date(2026, 4, 1),
        'DESCRIPTION': 'The BAF09 on our Yokohama to Szczecin corridor is 300 USD/40ST. '
                       'Our competitor offers the same route with BAF included. Can you match?',
        'CASE_TYPE': 'B_ESCALADE',
        'EXPECTED_DECISION': 'escalader',
        'EXPECTED_FINDING': '',
        'EXPECTED_ANSWER': 'Demande commerciale hors périmètre du support technique. '
                          'Pas de preuve d\'un écart dans les données. Escalade.'
    },
    # Ticket sans devis (description vide / floue → préciser)
    {
        'TICKET_ID': 'TK-SIM-010',
        'QUOTE_ID': None,
        'BOOKING_ID': None,
        'CREATED_DATE': date(2026, 4, 5),
        'DESCRIPTION': 'The price is wrong.',
        'CASE_TYPE': 'D_DESCRIPTION_VIDE',
        'EXPECTED_DECISION': 'preciser',
        'EXPECTED_FINDING': '',
        'EXPECTED_ANSWER': 'La description est trop vague. Merci de préciser la route, '
                          'la date et la référence du devis ou de la réservation.'
    },
])

print(f"TICKETS : {len(tickets)} tickets")
for ct in tickets.CASE_TYPE.unique():
    print(f"  {ct} : {(tickets.CASE_TYPE == ct).sum()}")
tickets[['TICKET_ID','QUOTE_ID','CASE_TYPE','EXPECTED_DECISION']]

### 4. Écriture dans Snowflake

In [ ]:
for name, data in [('QUOTES', quotes), ('BOOKINGS', bookings), ('TICKETS', tickets)]:
    session.write_pandas(
        data, name,
        database='PROJECT_DB', schema='PUBLIC',
        auto_create_table=True, overwrite=True,
    )
    print(f"{name} : {len(data)} lignes")

### 5. Vérification d'intégrité

Chaque devis doit pointer vers une ligne existante dans BAF09.
Chaque booking doit pointer vers un devis existant.

In [ ]:
# Vérif 1 : chaque QUOTE.DETAIL_UID existe dans PRICELINES_BAF09
baf_uids = set(
    r['DETAIL_UID'] for r in
    session.sql('SELECT DISTINCT DETAIL_UID FROM PROJECT_DB.PUBLIC.PRICELINES_BAF09').collect()
)
missing_uids = [q for q in quotes['DETAIL_UID'] if q not in baf_uids]
print(f"Devis avec detail_uid introuvable dans BAF09 : {len(missing_uids)}")
if missing_uids:
    print("  PROBLÈME :", missing_uids)
else:
    print("  OK : tous les devis pointent vers une priceline existante")

# Vérif 2 : chaque BOOKING.QUOTE_ID existe dans QUOTES
quote_ids = set(quotes['QUOTE_ID'])
missing_quotes = [b for b in bookings['QUOTE_ID'] if b not in quote_ids]
print(f"\nBookings avec quote_id introuvable : {len(missing_quotes)}")
if missing_quotes:
    print("  PROBLÈME :", missing_quotes)
else:
    print("  OK : toutes les réservations pointent vers un devis existant")

# Vérif 3 : chaque TICKET.QUOTE_ID (quand non null) existe dans QUOTES
tkt_qids = [t for t in tickets['QUOTE_ID'].dropna() if t not in quote_ids]
print(f"\nTickets avec quote_id introuvable : {len(tkt_qids)}")
if tkt_qids:
    print("  PROBLÈME :", tkt_qids)
else:
    print("  OK : tous les tickets avec devis pointent vers un devis existant")

# Vérif 4 : chaque TICKET.BOOKING_ID (quand non null) existe dans BOOKINGS
booking_ids = set(bookings['BOOKING_ID'])
tkt_bids = [t for t in tickets['BOOKING_ID'].dropna() if t not in booking_ids]
print(f"\nTickets avec booking_id introuvable : {len(tkt_bids)}")
if tkt_bids:
    print("  PROBLÈME :", tkt_bids)
else:
    print("  OK : tous les tickets avec booking pointent vers un booking existant")

In [ ]:
# Vérif 5 : vérifier que find_pricelines retrouve bien les cas attendus
import os, sys, importlib
parent = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent not in sys.path:
    sys.path.insert(0, parent)
import src.tools.pricelines
importlib.reload(src.tools.pricelines)
from src.tools.pricelines import find_pricelines

results = []
for _, q in quotes.iterrows():
    r = find_pricelines(
        session, q['POL_CODE'], q['POD_CODE'],
        q['QUOTE_DATE'].isoformat(),
        container_type=q['CONTAINER_TYPE'] if pd.notna(q['CONTAINER_TYPE']) else None
    )
    ok_evidence = (r['evidence'] == q['EXPECTED_EVIDENCE'])
    results.append({
        'QUOTE_ID': q['QUOTE_ID'],
        'CASE_TYPE': q['CASE_TYPE'],
        'nb_candidates': r['nb_candidates'],
        'evidence': r['evidence'],
        'expected_evidence': q['EXPECTED_EVIDENCE'],
        'OK': ok_evidence,
    })

results_df = pd.DataFrame(results)
print(f"{results_df['OK'].sum()} / {len(results_df)} devis avec evidence conforme")
results_df

### Résumé des cas fabriqués

| Cas | Scénario | Décision attendue | Evidence |
|-----|----------|-------------------|----------|
| A1 | MXVER→ESSCT, BAF modifié manuellement | répondre | True |
| A2 | ALDRZ→AEJEA, BAF modifié manuellement | répondre | True |
| A3 | ALDRZ→AEAJM, BAF modifié manuellement | répondre | True |
| A4 | MXVER→ESSCT, autre detail_uid, même écart | répondre | True |
| B1 | PKKHI→LVRIX, pas de version Manual | escalader | False |
| B2 | BDCGP→MAAGA, pas de version Manual | escalader | False |
| B3 | IDKUM→GBTIL, pas de version Manual | escalader | False |
| B4 | JPYOK→PLSZZ, pas de version Manual | escalader | False |
| C1 | AEJEA→AUADL, dry + reefer sans type | préciser | False |
| C2 | ALDRZ→ASPPG, dry + reefer sans type | préciser | False |